In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path
import numpy as np
import malaya_speech
from malaya_speech.model.clustering import StreamingKMeans

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
2025-09-28 13:55:06.715004: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759067706.724066    4360 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759067706.728440    4360 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759067706.733677    4360 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:175906770

In [2]:
df = pd.read_parquet('common-voice-22.parquet')

In [3]:
mapping = {}
for i in tqdm(range(len(df))):
    mapping[df['audio'].iloc[i]] = i
len(mapping)

100%|██████████| 8525319/8525319 [00:33<00:00, 253082.00it/s]


8525319

In [4]:
from datasets import load_dataset

ds = load_dataset("malaysia-ai/common_voice_22_0")

In [5]:
import os

def new_path(f):
    splitted = f.split('/')
    base_folder = splitted[0] + '_trim'
    splitted = '/'.join([base_folder] + splitted[1:])
    return splitted

In [6]:
train = ds['train'].to_pandas()
dev = ds['dev'].to_pandas()

In [7]:
train = train[train['down_votes'] <= 0]
dev = dev[dev['down_votes'] <= 0]
train.shape, dev.shape

((6423060, 13), (498409, 13))

In [8]:
concated = pd.concat([train, dev])
concated.to_parquet('concated-common-voice-22.parquet')

In [9]:
locales = [s for s in concated['locale'].unique().tolist() if len(s) > 1]
len(locales)

132

In [10]:
# import torch

# streaming = StreamingKMeans(0.3)

# results_similarity = {}
# for i in tqdm(range(len(filtered))):
#     index = mapping[new_path(filtered['path'].iloc[i])]
#     v_f = f'common-voice-22/{index}.npy'
#     if not os.path.exists(v_f):
#         continue
#     try:
#         v = np.load(v_f)
#         results_similarity[filtered['path'].iloc[i]] = streaming.streaming(v)
#     except Exception as e:
#         print(e)
#         pass

In [11]:
def loop(locales):
    import faiss
    
    locales, _ = locales
    df = pd.read_parquet('concated-common-voice-22.parquet')
    data = []
    for l in locales:

        d = 192
        index = faiss.IndexFlatL2(d)
        
        centroids = []
        
        def assign(x, threshold=0.1):
            if len(centroids) == 0:
                centroids.append(x)
                index.add(np.array([x], dtype=np.float32))
                return 0
            
            D, I = index.search(np.array([x], dtype=np.float32), 1)
            if D[0][0] > threshold:
                centroids.append(x)
                index.add(np.array([x], dtype=np.float32))
                return len(centroids)-1
            else:
                return I[0][0]
                
        filtered = df[df['locale'] == l]
        for i in tqdm(range(len(filtered))):
            index_ = mapping[new_path(filtered['path'].iloc[i])]
            v_f = f'common-voice-22/{index_}.npy'
            if not os.path.exists(v_f):
                continue
            try:
                v = np.load(v_f)
                data.append({
                    'path': filtered['path'].iloc[i],
                    'speaker': assign(v),
                    'locale': l
                })
            except Exception as e:
                print(e)
                pass
    return data

In [12]:
# data = loop((locales[:2], 0))

In [13]:
data = multiprocessing(locales, loop, cores = 20)

100%|██████████| 33852/33852 [00:13<00:00, 2537.34it/s]


In [14]:
len(data)

6921469

In [17]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'path': 'audio/en/train/common_voice_en_23679703.mp3',
 'speaker': 0,
 'locale': 'en'}

In [18]:
dataset.push_to_hub('malaysia-ai/common_voice_22_0', 'pseudospeaker')

Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00,  6.71ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  23%|██▎       | 11.0MB / 47.3MB, 1.12MB/s  
Processing Files (0 / 1):  99%|█████████▉| 46.9MB / 47.3MB, 4.69MB/s  
Processing Files (1 / 1): 100%|██████████| 47.3MB / 47.3MB, 4.66MB/s  
Processing Files (1 / 1): 100%|██████████| 47.3MB / 47.3MB, 4.73MB/s  
New Data Upload: 100%|██████████| 47.3MB / 47.3MB, 4.73MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:12<00:00, 12.15s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/common_voice_22_0/commit/0388c8b7c80263581630ffa725297918d72aa4d0', commit_message='Upload dataset', commit_description='', oid='0388c8b7c80263581630ffa725297918d72aa4d0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/common_voice_22_0', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/common_voice_22_0'), pr_revision=None, pr_num=None)

In [22]:
speaker_mapping = {}
for d in tqdm(data):
    speaker_mapping[d['path']] = d['speaker']

100%|██████████| 6921469/6921469 [00:03<00:00, 2033180.19it/s]


In [28]:
processed = []
for i in tqdm(range(len(concated))):
    try:
        t = concated['sentence'].iloc[i].strip()
        if len(t) < 2:
            continue
        l = concated['locale'].iloc[i]
        s = speaker_mapping[concated['path'].iloc[i]]
        processed.append({
            'audio_filename': new_path(concated['path'].iloc[i]),
            'text': t,
            'speaker': f"common_voice_22_{l}_{s}"
        })
    except:
        pass

100%|██████████| 6921469/6921469 [01:47<00:00, 64590.10it/s]


In [29]:
len(processed)

6921399

In [30]:
from datasets import Dataset

dataset = Dataset.from_list(processed)
dataset[0]

{'audio_filename': 'audio_trim/en/train/common_voice_en_23679703.mp3',
 'text': 'He directed four episodes of the series from seasons two through five.',
 'speaker': 'common_voice_22_en_0'}

In [33]:
dataset[-1]

{'audio_filename': 'audio_trim/eo/dev/common_voice_eo_28226204.mp3',
 'text': 'La urbodomo situas laŭ placo.',
 'speaker': 'common_voice_22_eo_3687'}

In [31]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'common-voice-22')

Creating parquet from Arrow format: 100%|██████████| 4/4 [00:00<00:00,  5.35ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  22%|██▏       | 27.7MB /  123MB,   ???B/s  
Processing Files (0 / 1):  99%|█████████▉|  122MB /  123MB,  472MB/s  
Processing Files (0 / 1): 100%|█████████▉|  123MB /  123MB,  119MB/s  
Processing Files (1 / 1): 100%|██████████|  123MB /  123MB, 95.7MB/s  
Processing Files (1 / 1): 100%|██████████|  123MB /  123MB, 79.8MB/s  
New Data Upload: 100%|██████████|  123MB /  123MB, 79.8MB/s  
Creating parquet from Arrow format: 100%|██████████| 4/4 [00:00<00:00,  5.57ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  21%|██        | 24.3MB /  118MB,   ???B/s  
Processing Files (0 / 1):  92%|█████████▏|  109MB /  118MB,  422MB/s  
Processing Files (0 / 1):  99%|█████████▉|  117MB /  118MB,  232MB/s  
Processing Files (1 / 1): 100%|██████████|  118MB /  118MB, 93.6MB/s  
P

CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/38d2a7dadcda4541bffef5aa3d9a119187f1f787', commit_message='Upload dataset', commit_description='', oid='38d2a7dadcda4541bffef5aa3d9a119187f1f787', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)